# NB_lab_test_trends

Builds `adb_caresync_smd.gold.lab_test_trends` — one row per test type per month with abnormal and critical rates and month-over-month trend metrics.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync_smd.gold.lab_test_trends AS

WITH monthly_base AS (
  SELECT
    lr.test_name,
    YEAR(lr.test_date)                                           AS year,
    MONTH(lr.test_date)                                          AS month,
    DATE_TRUNC('month', lr.test_date)                            AS month_start_date,
    COUNT(*)                                                     AS total_tests,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)    AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)    AS critical_test_count
  FROM adb_caresync_smd.silver.lab_results lr
  GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date), DATE_TRUNC('month', lr.test_date)
), enriched AS (
  SELECT
    mb.test_name,
    mb.year,
    mb.month,
    c.month_name,
    c.quarter_name,
    mb.total_tests,
    mb.abnormal_test_count,
    mb.critical_test_count,
    ROUND(mb.abnormal_test_count / NULLIF(mb.total_tests, 0) * 100, 2)
                                                                AS abnormal_rate_pct,
    ROUND(mb.critical_test_count / NULLIF(mb.total_tests, 0) * 100, 2)
                                                                AS critical_rate_pct
  FROM monthly_base mb
  LEFT JOIN adb_caresync_smd.silver.calendar c
    ON c.date = mb.month_start_date
)

SELECT
  test_name,
  year,
  month,
  month_name,
  quarter_name,
  total_tests,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month)
                                                                AS prev_month_abnormal_rate_pct,
  ROUND(
    abnormal_rate_pct - LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month),
    2
  )                                                             AS abnormal_rate_change_pct
FROM enriched;

In [0]:
updated_df = spark.table("adb_caresync_smd.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))